# RVC — Pipeline Đầu Cuối (Google Colab Version)

Pipeline đầy đủ: **Chuẩn Bị → Training → Inference → Đánh Giá**, tối ưu cho Google Colab.

```
[LibriTTS / Drive] → Chuẩn bị data → Preprocess → Trích F0 + Hubert → FAISS Index
                                                             ↓
         [Vòng lặp — chạy nhiều lần nếu COMPARE_MULTIPLE_MODELS=True]
         ┌──────────────────────────────────────────────────────────────┐
         │  Train đến epoch_target (continue từ checkpoint trước)       │
         │  → Export model_<N>ep.pth                                    │
         │  → Inference source_test → converted_output/model_<N>/       │
         │  → Đánh giá: Speaker Similarity + MOS + WER/CER              │
         └──────────────────────────────────────────────────────────────┘
                                                             ↓
                    Bảng tổng hợp + Biểu đồ so sánh các model
```

---

## Thứ tự chạy (lần đầu)

| # | Cell | Tác dụng |
|---|------|----------|
| 1 | Mount Drive + GPU | Kết nối Drive, kiểm tra GPU |
| 2 | Clone GitHub | Tải mã nguồn RVC từ GitHub |
| 2b | Cài thư viện | Cài toàn bộ dependencies |
| 3 | Symlink Drive | Checkpoint/weights tự lưu vào Drive |
| 4 | Cấu hình | Đặt tham số trước khi chạy pipeline |
| 5 | Kiểm tra môi trường | Xác minh weights, GPU, FFmpeg |
| 6 | Thư viện Python | Kiểm tra & cài bổ sung |
| 7 | Import & Setup | Import module, setup runtime |
| 8 | Load dữ liệu đã chuẩn bị | Đọc manifest từ data_check_and_prepare.ipynb |
| 9 | Preprocess & Feature extraction | Resample, trích F0 + HuBERT |
| 10 | FAISS Index | Xây dựng index một lần |
| 11 | Load eval models | SpeechBrain, Whisper, UTMOS |
| 12 | **Vòng lặp chính** | Train → Export → Inference → Đánh giá |
| 13 | Tổng hợp kết quả | Bảng + biểu đồ so sánh |

> **Yêu cầu:** Chạy `data_check_and_prepare.ipynb` **trước** notebook này để chuẩn bị data và tải weights.

> **Resume sau khi Colab reset:** Chạy lại Cell 1→3, Cell 2b, rồi tiếp tục từ Cell 4.
> Training tự resume từ checkpoint cuối trên Drive.


## Cell 1 — Mount Google Drive & Kiểm tra GPU

**Mục đích:** Kết nối Google Drive và xác nhận runtime có GPU trước khi làm bất cứ việc gì.

**Tại sao cần Drive?**
Colab session bị reset sau mỗi 12 giờ hoặc khi ngắt kết nối. Toàn bộ dữ liệu, checkpoint,
và kết quả sẽ bị mất nếu chỉ lưu trong `/content`.
Bằng cách mount Drive, mọi thứ được ghi thẳng vào `MyDrive/rvc_project/` và tồn tại vĩnh viễn.

**Tại sao cần GPU?**
Các bước `trích HuBERT features`, `training`, và `inference` đều yêu cầu CUDA.
Nếu không có GPU, cell này sẽ dừng ngay và nhắc bạn đổi runtime.

> **Trước khi chạy:** `Runtime → Change runtime type → GPU (T4 hoặc A100)`

**Biến quan trọng được tạo ra:**
- `DRIVE_BASE = '/content/drive/MyDrive/rvc_project'` — thư mục gốc trên Drive


In [1]:
from google.colab import drive
import os, torch

drive.mount('/content/drive')

if not torch.cuda.is_available():
    print("CANH BAO: Khong co GPU. Training se rat cham (CPU mode).")
    print("Khuyen nghi: Runtime -> Change runtime type -> GPU (T4 hoac A100)")
else:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | {props.total_memory/1024**3:.1f} GB VRAM | CUDA {torch.version.cuda}")

DRIVE_BASE = '/content/drive/MyDrive/rvc_project'
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f"Drive base: {DRIVE_BASE}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4 | 14.6 GB VRAM | CUDA 12.8
Drive base: /content/drive/MyDrive/rvc_project


## Cell 2 — Clone Repo từ GitHub

**Mục đích:** Tải mã nguồn RVC (thư viện `infer/`, `configs/`, `training_pipeline/`, v.v.)
từ repository GitHub về `/content/compare_rvc`.

**Chiến lược:**
- Nếu thư mục `.git` đã tồn tại (session cũ chưa reset hẳn) → chỉ `git pull` để cập nhật.
- Nếu chưa có → `git clone` từ đầu (thường mất 10–30 giây).

**Tại sao không lưu code vào Drive?**
Code clone nhanh (~vài chục MB). Những thứ quan trọng cần persist
(checkpoint, weights, data) sẽ được symlink sang Drive ở Cell 3.
Giữ code trong `/content` giúp đọc/ghi nhanh hơn Drive.

**Sau cell này:** Working directory chuyển sang `/content/compare_rvc`.


In [2]:
import os, subprocess

REPO_URL = 'https://github.com/thanhNgan13/compare_rvc.git'
REPO_DIR = '/content/compare_rvc'

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('Repo da co - cap nhat...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    print(f'Clone {REPO_URL} ...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')
print('\nCau truc thu muc:')
for item in sorted(os.listdir('.')):
    print(f"  {item}{'/' if os.path.isdir(item) else ''}")


Repo da co - cap nhat...
Working dir: /content/compare_rvc

Cau truc thu muc:
  .git/
  .gitignore
  Danh_gia_RVC_Voice_Conversion.ipynb
  RVC_infer_huong_dan_chi_tiet.ipynb
  RVC_pipeline_colab.ipynb
  RVC_pipeline_hoan_chinh.ipynb
  RVC_training_huong_dan_chi_tiet.ipynb
  TEMP/
  assets/
  configs/
  data_check_and_prepare.ipynb
  datasets/
  infer/
  logs/
  requirements.txt
  tools/
  training_pipeline/


## Cell 2b — Cài Đặt Toàn Bộ Thư Viện từ `requirements.txt`

**Mục đích:** Cài đặt tất cả dependency của dự án từ file `requirements.txt`
trong repo vừa clone về, đảm bảo môi trường Colab khớp với môi trường phát triển.

**Thứ tự xử lý:**

1. **Cài `aria2` và `ffmpeg` qua `apt`** — đây là system tool, không phải Python package,
   nên không thể cài bằng pip.

2. **Downgrade pip xuống < 24.1** — bắt buộc trước khi cài `fairseq==0.12.2`.
   `fairseq` phụ thuộc `omegaconf 2.0.x` có metadata không chuẩn;
   pip ≥ 24.1 từ chối resolve dependency này.

3. **Lọc `requirements.txt` trước khi cài:**
   - Bỏ `aria2` (đã cài ở bước 1)
   - Bỏ `onnxruntime` / `onnxruntime-gpu` dòng gốc (platform condition không hoạt
     đúng trên Colab) → thay bằng `onnxruntime-gpu` phiên bản tương thích CUDA của Colab
   - Bỏ các dòng comment

4. **`pip install -r requirements_colab.txt`** — cài toàn bộ còn lại.

> **Thời gian:** ~5–10 phút. Nhiều package có sẵn wheel cho Python 3.10
> nên không cần build từ source.


In [3]:
# import subprocess, sys
# from pathlib import Path


# def run(cmd):
#     print(f'  $ {cmd}')
#     r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
#     if r.returncode != 0:
#         print('FAILED:')
#         print((r.stdout + r.stderr)[-4000:])
#         raise subprocess.CalledProcessError(r.returncode, cmd)


# def run_with_fallback(pinned, latest):
#     try:
#         run(f'pip install -q {pinned}')
#     except subprocess.CalledProcessError:
#         print('  Pin that bai, thu lai voi phien ban moi hon...')
#         run(f'pip install -q {latest}')


# # 1. System packages
# print('=== 1/3  System packages ===')
# run('apt-get install -qq -y aria2 ffmpeg')

# # 2. Downgrade pip truoc khi cai fairseq
# print('\n=== 2/3  Downgrade pip ===')
# run('pip install -q "pip>=23.2,<24.1"')

# # 3. Loc va cai requirements.txt
# print('\n=== 3/3  Cai tu requirements.txt ===')
# req_path = Path('/content/compare_rvc/requirements.txt')
# if not req_path.exists():
#     raise FileNotFoundError('requirements.txt khong tim thay. Chay lai Cell 2 truoc.')

# # Packages co the khong co wheel cho Python hien tai -> cai rieng voi fallback
# PINNED_WITH_FALLBACK = {
#     'numba==0.56.4':    'numba',
#     'llvmlite==0.39.0': 'llvmlite',
#     'numpy==1.23.5':    'numpy',
#     'librosa==0.9.1':   'librosa',
#     'faiss-cpu==1.7.3': 'faiss-cpu',
#     'pyworld==0.3.2':   'pyworld',
# }

# lines_out, fallback_needed = [], []
# for line in req_path.read_text(encoding='utf-8').splitlines():
#     s = line.strip()
#     if not s or s.startswith('#'):
#         continue
#     if s.lower() == 'aria2':
#         continue
#     if s.startswith('onnxruntime'):
#         continue
#     if s in PINNED_WITH_FALLBACK:
#         fallback_needed.append(s)
#         continue
#     lines_out.append(s)
# lines_out.append('onnxruntime-gpu')

# colab_req = Path('/tmp/requirements_colab.txt')
# colab_req.write_text('\n'.join(lines_out), encoding='utf-8')
# print(f'  {len(lines_out)} packages chinh')
# run(f'pip install -r {colab_req}')

# print('\n=== Cai packages co pin version co the khong tuong thich ===')
# for pinned in fallback_needed:
#     run_with_fallback(pinned, PINNED_WITH_FALLBACK[pinned])


# # Cai them cac package danh gia khong co trong requirements.txt
# print('\n=== Extra: packages danh gia ===')
# run('pip install -q jiwer speechbrain transformers')

# print('\nCai dat hoan tat.')


## Cell 3 — Tạo Symlink: Checkpoint & Weights → Google Drive

**Mục đích:** Ánh xạ các thư mục quan trọng trong repo sang Google Drive
bằng symbolic link, để mọi thứ được ghi trực tiếp lên Drive mà không cần copy thủ công.

**Cơ chế hoạt động:**
```
/content/compare_rvc/logs/           → /MyDrive/rvc_project/logs/
/content/compare_rvc/assets/weights/ → /MyDrive/rvc_project/model_weights/
/content/compare_rvc/assets/hubert/  → /MyDrive/rvc_project/assets_cache/hubert/
/content/compare_rvc/assets/rmvpe/   → /MyDrive/rvc_project/assets_cache/rmvpe/
/content/compare_rvc/assets/pretrained_v2/ → /MyDrive/rvc_project/assets_cache/pretrained_v2/
```

**Lợi ích:**
- Training pipeline viết checkpoint vào `logs/` như bình thường → checkpoint tự động lên Drive.
- Khi Colab reset, clone lại repo, tạo lại symlink → training tiếp tục từ checkpoint cuối.
- Weights (HuBERT, RMVPE, pretrained) chỉ cần tải 1 lần, các session sau dùng lại.

**Xử lý nội dung cũ:** Nếu thư mục local đã có file (ví dụ repo có sẵn `assets/hubert` rỗng),
script di chuyển nội dung sang Drive trước khi tạo symlink, tránh mất dữ liệu.


In [4]:
import os, shutil
from pathlib import Path

REPO  = Path('/content/compare_rvc')
DRIVE = Path(DRIVE_BASE)

SYMLINKS = {
    'logs'                 : DRIVE / 'logs',
    'assets/weights'       : DRIVE / 'model_weights',
    'assets/hubert'        : DRIVE / 'assets_cache/hubert',
    'assets/rmvpe'         : DRIVE / 'assets_cache/rmvpe',
    'assets/pretrained_v2' : DRIVE / 'assets_cache/pretrained_v2',
}

for rel, drive_path in SYMLINKS.items():
    drive_path.mkdir(parents=True, exist_ok=True)
    local = REPO / rel
    local.parent.mkdir(parents=True, exist_ok=True)

    if local.is_symlink():
        local.unlink()
    elif local.is_dir():
        for item in local.iterdir():
            dst = drive_path / item.name
            if not dst.exists():
                shutil.move(str(item), str(dst))
        shutil.rmtree(str(local))

    local.symlink_to(drive_path)
    print(f'{str(local):<55} -> {drive_path}')

print('\nSymlinks OK — checkpoint va weights tu luu vao Drive.')


/content/compare_rvc/logs                               -> /content/drive/MyDrive/rvc_project/logs
/content/compare_rvc/assets/weights                     -> /content/drive/MyDrive/rvc_project/model_weights
/content/compare_rvc/assets/hubert                      -> /content/drive/MyDrive/rvc_project/assets_cache/hubert
/content/compare_rvc/assets/rmvpe                       -> /content/drive/MyDrive/rvc_project/assets_cache/rmvpe
/content/compare_rvc/assets/pretrained_v2               -> /content/drive/MyDrive/rvc_project/assets_cache/pretrained_v2

Symlinks OK — checkpoint va weights tu luu vao Drive.


## Cell 4 — Cấu Hình Pipeline

**Mục đích:** Khai báo tất cả tham số điều khiển toàn bộ pipeline.
Đây là **cell duy nhất cần sửa** trước khi chạy; các cell còn lại đọc từ đây.

---

### Nhóm 1 — Cờ chính
- `COMPARE_MULTIPLE_MODELS`: `True` = train và so sánh nhiều model với số epoch khác nhau;
  `False` = chỉ train 1 model duy nhất.
- `EPOCH_LIST`: danh sách mốc epoch — training **continue** từ checkpoint, không train lại từ đầu.
  Ví dụ `[50, 100, 200]`: lần 1 train 0→50, lần 2 tiếp tục 50→100, lần 3 tiếp tục 100→200.

### Nhóm 2 — Dataset
- `DATASET_ROOT`: đường dẫn LibriTTS trên Drive (khớp với Cell 5).
- `TARGET_TRAIN_SECONDS` / `TARGET_EVAL_SECONDS`: phân chia audio của target speaker.
- `SOURCE_TEST_COUNT`: số câu của source speaker dùng để inference.

### Nhóm 3 — Training
- `BATCH_SIZE = 8`: phù hợp T4 (15 GB VRAM); nâng lên 16 nếu dùng A100.
- `NUM_PROCESSES = 2`: số CPU core cho preprocess (Colab thường có 2 core).
- `F0_METHOD_TRAIN = 'rmvpe'`: phương pháp trích F0 — `rmvpe` chính xác nhất.

### Nhóm 4 — Inference
- `INFER_F0_UP_KEY`: dịch tông (semitone). `0` = giữ nguyên; `+12/-12` = lên/xuống 1 quãng tám.
  Khuyến nghị `+12` nếu chuyển giọng nam → nữ.
- `INFER_INDEX_RATE`: tỉ lệ dùng FAISS index (0–1). Cao → giống giọng target hơn nhưng có thể bị artifact.

### Nhóm 5 — Đánh giá
- `ASR_MODEL_ID`: model Whisper cho tính WER/CER. `whisper-small` cân bằng tốc độ/độ chính xác.
- `MOS_BACKEND = 'utmos_torchhub'`: dùng UTMOS để dự đoán điểm chất lượng âm thanh (1–5).

### Nhóm 6 — Đường dẫn
Mọi output (prepared data, kết quả, converted audio) đều lưu vào `MyDrive/rvc_project/eval_project/`.


In [5]:
from pathlib import Path

COMPARE_MULTIPLE_MODELS = True
EPOCH_LIST              = [50, 100, 200]
SINGLE_MODEL_EPOCHS     = 100

DRIVE_BASE           = '/content/drive/MyDrive/rvc_project'
DATASET_ROOT         = Path(DRIVE_BASE) / 'eval_project/raw_data/LibriTTS'
ARCHIVE_DIR          = Path(DRIVE_BASE) / 'downloads'
TARGET_SPLIT         = 'train-clean-100'
SOURCE_SPLIT         = 'test-clean'
TARGET_SPEAKER_ID    = None
SOURCE_SPEAKER_ID    = None
TARGET_TRAIN_SECONDS = 10 * 60
TARGET_EVAL_SECONDS  = 2  * 60
SOURCE_TEST_COUNT    = 10
MIN_SOURCE_DURATION  = 2.0
MAX_SOURCE_DURATION  = 12.0
PREPARED_AUDIO_SR    = None
RESET_PREPARED_DATA  = False

EXPERIMENT_NAME    = 'rvc_experiment'
SAMPLE_RATE_LABEL  = '40k'
RVC_VERSION        = 'v2'
IF_F0              = True
F0_METHOD_TRAIN    = 'rmvpe'
NUM_PROCESSES      = 2
GPU_DEVICES        = '0'
BATCH_SIZE         = 8
SAVE_EVERY_EPOCH   = 10
SKIP_INDEX         = False

INFER_SPEAKER_ID    = 0
INFER_F0_UP_KEY     = 0
INFER_F0_METHOD     = 'rmvpe'
INFER_INDEX_RATE    = 0.75
INFER_FILTER_RADIUS = 3
INFER_RESAMPLE_SR   = 0
INFER_RMS_MIX_RATE  = 0.25
INFER_PROTECT       = 0.33

ASR_MODEL_ID      = 'openai/whisper-small'
ASR_LANGUAGE      = 'english'
RUN_PREDICTED_MOS = True
MOS_BACKEND       = 'utmos_torchhub'

PROJECT_DIR    = Path(DRIVE_BASE) / 'eval_project'
PREPARED_DIR   = PROJECT_DIR / 'prepared_data'
CONVERTED_ROOT = PROJECT_DIR / 'converted_output'
RESULTS_DIR    = PROJECT_DIR / 'results'

for _d in [PROJECT_DIR, PREPARED_DIR, CONVERTED_ROOT, RESULTS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

RANDOM_SEED          = 42
EPOCH_LIST_EFFECTIVE = sorted(EPOCH_LIST) if COMPARE_MULTIPLE_MODELS else [SINGLE_MODEL_EPOCHS]

print(f'So sanh nhieu model : {COMPARE_MULTIPLE_MODELS}')
print(f'Danh sach epoch     : {EPOCH_LIST_EFFECTIVE}')
print(f'Batch size / GPU    : {BATCH_SIZE} / {GPU_DEVICES}')
print(f'Dataset root        : {DATASET_ROOT}')
print(f'Project dir         : {PROJECT_DIR}')


So sanh nhieu model : True
Danh sach epoch     : [50, 100, 200]
Batch size / GPU    : 8 / 0
Dataset root        : /content/drive/MyDrive/rvc_project/eval_project/raw_data/LibriTTS
Project dir         : /content/drive/MyDrive/rvc_project/eval_project


## Cell 5 — Kiểm Tra Môi Trường

**Mục đích:** Xác minh rằng tất cả điều kiện tiên quyết đều đã sẵn sàng
trước khi bắt đầu pipeline tốn thời gian.

**Những gì được kiểm tra:**

1. **Cấu trúc thư mục** — `infer/`, `configs/`, `training_pipeline/`, `assets/`, `logs/`
   phải tồn tại. Nếu thiếu → Cell 2 (clone) hoặc Cell 3 (symlink) chưa chạy.

2. **GPU** — In tên GPU và VRAM. Pipeline có thể chạy trên CPU nhưng sẽ rất chậm (không khuyến nghị).

3. **Model weights** — Kiểm tra 4 file weights thiết yếu.
   Nếu thiếu → chạy data_check_and_prepare.ipynb.

4. **FFmpeg** — Cần cho encode/decode audio. Nếu thiếu → chạy lại Cell 2b.

> Chạy cell này bất cứ lúc nào muốn kiểm tra trạng thái môi trường.


In [6]:
import subprocess, pathlib, torch

CWD = pathlib.Path.cwd().resolve()
print('Working directory:', CWD)

required_dirs = ['infer', 'configs', 'training_pipeline', 'assets', 'logs']
missing = [d for d in required_dirs if not (CWD / d).is_dir()]
print('Thu muc:', 'OK' if not missing else f'THIEU {missing}')

print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p2 = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p2.name} | {p2.total_memory/1024**3:.1f} GB VRAM')

weights = [
    ('assets/hubert/hubert_base.pt',    'HuBERT'),
    ('assets/rmvpe/rmvpe.pt',           'RMVPE'),
    ('assets/pretrained_v2/f0G40k.pth', 'Pretrained G'),
    ('assets/pretrained_v2/f0D40k.pth', 'Pretrained D'),
]
for path, name in weights:
    f = CWD / path
    status = f'{f.stat().st_size/1024**2:.0f} MB' if f.exists() else 'THIEU'
    print(f'  {name}: {status}')

try:
    r = subprocess.run(['ffmpeg', '-version'], capture_output=True, timeout=5)
    ver = r.stdout.decode(errors='ignore').splitlines()[0][:50]
    print(f'FFmpeg: {ver}')
except FileNotFoundError:
    print('FFmpeg: KHONG TIM THAY - chay lai Cell 4')


Working directory: /content/compare_rvc
Thu muc: OK
PyTorch 2.11.0+cu128 | CUDA True
  GPU 0: Tesla T4 | 14.6 GB VRAM
  HuBERT: 181 MB
  RMVPE: 173 MB
  Pretrained G: 70 MB
  Pretrained D: 136 MB
FFmpeg: ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c


## Cell 6 — Kiểm Tra & Cài Bổ Sung Thư Viện Python

**Mục đích:** Đảm bảo tất cả package Python cần thiết đã được import được.
Cell 2b đã cài hầu hết; cell này là lớp kiểm tra bổ sung —
chỉ gọi `pip install` cho những gì còn thiếu, không cài lại toàn bộ.

**Danh sách kiểm tra:** `numpy`, `pandas`, `tqdm`, `soundfile`, `librosa`,
`scikit-learn`, `matplotlib`, `jiwer`, `transformers`, `speechbrain`.

> Cell này an toàn để chạy nhiều lần — nếu tất cả đã có sẽ in `OK` và thoát nhanh.


In [7]:
import importlib.util, subprocess, sys

packages = {
    'numpy':        'numpy',
    'pandas':       'pandas',
    'tqdm':         'tqdm',
    'soundfile':    'soundfile',
    'librosa':      'librosa',
    'sklearn':      'scikit-learn',
    'matplotlib':   'matplotlib',
    'jiwer':        'jiwer',
    'transformers': 'transformers',
    'speechbrain':  'speechbrain',
}

for name, pip_name in packages.items():
    if importlib.util.find_spec(name) is None:
        print(f'Cai {pip_name}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])
    else:
        print(f'OK: {name}')


OK: numpy
OK: pandas
OK: tqdm
OK: soundfile
OK: librosa
OK: sklearn
OK: matplotlib
OK: jiwer
OK: transformers
OK: speechbrain


## Cell 7 — Import Toàn Bộ & Thiết Lập Môi Trường Runtime

**Mục đích:** Import tất cả module và thiết lập các biến môi trường
mà RVC inference engine yêu cầu.

**Các thiết lập quan trọng:**

- **`sys.path`:** Thêm `/content/compare_rvc` vào đầu `sys.path`
  để `from infer.modules...` và `from training_pipeline...` tìm được module.

- **Biến môi trường RVC:** `weight_root`, `index_root`, `rmvpe_root` —
  đây là các path mà module `infer/` dùng để tìm file weights;
  phải set trước khi import bất kỳ module nào từ `infer/`.

- **`STANDALONE_ROOT`:** Biến toàn cục trỏ đến `/content/compare_rvc`,
  được dùng xuyên suốt pipeline để build đường dẫn tuyệt đối.

- **`DEVICE`:** `'cuda'` nếu có GPU, `'cpu'` nếu không.
  Tất cả model inference và embedding đều dùng biến này.

> Phải chạy cell này **sau** Cell 4 (config) vì cần `RANDOM_SEED`.


In [8]:
import os, re, sys, time, json, random, shutil, unicodedata
from pathlib import Path
from typing import List, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import soundfile as sf
import librosa
import torch
import torchaudio
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import jiwer

STANDALONE_ROOT = Path('/content/compare_rvc').resolve()
os.chdir(STANDALONE_ROOT)
if str(STANDALONE_ROOT) not in sys.path:
    sys.path.insert(0, str(STANDALONE_ROOT))

os.environ.setdefault('weight_root',        'assets/weights')
os.environ.setdefault('index_root',         'logs')
os.environ.setdefault('outside_index_root', 'assets/indices')
os.environ.setdefault('rmvpe_root',         'assets/rmvpe')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('STANDALONE_ROOT:', STANDALONE_ROOT)
print('DEVICE         :', DEVICE)


STANDALONE_ROOT: /content/compare_rvc
DEVICE         : cuda


## Cell 8 — Load Dữ Liệu Đã Chuẩn Bị

Đọc manifest CSV từ `data_check_and_prepare.ipynb`. Nếu chưa có, hãy chạy notebook đó trước.

In [9]:
import pandas as pd
from pathlib import Path

manifest_files = {
    'Target Train': PREPARED_DIR / 'target_train_manifest.csv',
    'Target Eval':  PREPARED_DIR / 'target_eval_manifest.csv',
    'Source Test':  PREPARED_DIR / 'source_test_manifest.csv',
}

missing = [k for k, v in manifest_files.items() if not v.exists()]
if missing:
    raise FileNotFoundError(
        'Thieu manifest: ' + str(missing) +
        '. Chay data_check_and_prepare.ipynb truoc khi chay notebook nay.'
    )

tt = pd.read_csv(manifest_files['Target Train'])
te = pd.read_csv(manifest_files['Target Eval'])
st = pd.read_csv(manifest_files['Source Test'])

_target_id = str(tt['speaker_id'].iloc[0])
_source_id = str(st['speaker_id'].iloc[0])

print(f'Target speaker : {_target_id}')
print(f'Source speaker : {_source_id}')
print()
print(f'Target Train : {len(tt)} files | {tt["duration_sec"].sum()/60:.1f} phut')
print(f'Target Eval  : {len(te)} files | {te["duration_sec"].sum()/60:.1f} phut')
print(f'Source Test  : {len(st)} files | {st["duration_sec"].sum()/60:.1f} phut')

for subdir in [PREPARED_DIR / 'target_train',
               PREPARED_DIR / 'target_eval',
               PREPARED_DIR / 'source_test']:
    n = len(list(subdir.glob('*.wav'))) if subdir.exists() else 0
    print(f'  {subdir.name:<20} {n} .wav files')

print()
print('Du lieu san sang cho pipeline.')


Target speaker : 40
Source speaker : 1089

Target Train : 94 files | 10.1 phut
Target Eval  : 20 files | 2.1 phut
Source Test  : 10 files | 0.8 phut
  target_train         94 .wav files
  target_eval          20 .wav files
  source_test          10 .wav files

Du lieu san sang cho pipeline.


## Cell 9 — Khởi Tạo Training: Preprocess & Trích Features

**Mục đích:** Chuẩn bị dữ liệu training theo định dạng mà RVC yêu cầu,
bao gồm hai bước nặng nhất (chỉ chạy một lần và có cache):

### Bước A — Preprocess
Resample audio về sample rate của model (40kHz) và lưu vào
`logs/rvc_experiment/0_gt_wavs/`. Cell kiểm tra xem thư mục này đã có file chưa;
nếu có → bỏ qua.

### Bước B — Trích F0 + HuBERT Features
Đây là bước tốn thời gian nhất trong setup:
- **F0 (pitch):** Dùng RMVPE để trích cao độ từng frame → lưu vào `2a_f0/` và `2b-f0nsf/`.
- **HuBERT features:** Load model `hubert_base.pt`, encode từng clip thành vector 768 chiều
  → lưu vào `3_feature768/` (với RVC v2).

Cell kiểm tra `3_feature768/` đã có `.npy` chưa; nếu có → bỏ qua.

**Cuối cell:** Kiểm tra số file `gt_wavs` == số file `features`.
Nếu không khớp → preprocessing có thể bị lỗi một phần.

> **Thời gian:** Với 10 phút audio (40kHz, ~14,400 frame),
> bước này mất khoảng 15–30 phút trên T4.


In [10]:
import logging
logging.basicConfig(level=logging.WARNING, format='%(levelname)s | %(message)s')

from training_pipeline.setup_env import bootstrap
from training_pipeline.params import TrainingParams
from training_pipeline import steps as train_steps

root, config = bootstrap()
print(f'Device: {config.device} | FP16: {config.is_half}')

p = TrainingParams(
    experiment_name   = EXPERIMENT_NAME,
    trainset_dir      = str(PREPARED_DIR / 'target_train'),
    sample_rate_label = SAMPLE_RATE_LABEL,
    version           = RVC_VERSION,
    if_f0             = IF_F0,
    f0_method         = F0_METHOD_TRAIN,
    num_processes     = NUM_PROCESSES,
    gpu_devices_train = GPU_DEVICES,
    total_epochs      = max(EPOCH_LIST_EFFECTIVE),
    save_every_epoch  = SAVE_EVERY_EPOCH,
    batch_size        = BATCH_SIZE,
    skip_index        = SKIP_INDEX,
    infer_weight_name = 'temp_model',
    extract_infer_pth = False,
    speaker_id        = 0,
)

exp_dir          = STANDALONE_ROOT / 'logs' / EXPERIMENT_NAME
gt_wavs_existing = list((exp_dir / '0_gt_wavs').glob('*.wav')) if (exp_dir / '0_gt_wavs').exists() else []

if gt_wavs_existing:
    print(f'Preprocess: da co ({len(gt_wavs_existing)} clips) - bo qua')
else:
    print('Preprocess...')
    train_steps.step_preprocess(STANDALONE_ROOT, config, p)
    gt_wavs_existing = list((exp_dir / '0_gt_wavs').glob('*.wav'))
    print(f'Preprocess xong: {len(gt_wavs_existing)} clips')

feature_dir_name  = '3_feature768' if RVC_VERSION == 'v2' else '3_feature256'
feature_dir       = exp_dir / feature_dir_name
features_existing = list(feature_dir.glob('*.npy')) if feature_dir.exists() else []

if features_existing:
    print(f'Features: da co ({len(features_existing)} files) - bo qua')
else:
    print('Trich F0 + HuBERT features...')
    train_steps.step_extract_f0_and_features(STANDALONE_ROOT, config, p)
    features_existing = list(feature_dir.glob('*.npy'))
    print(f'Trich xong: {len(features_existing)} feature files')

print(f'gt_wavs: {len(gt_wavs_existing)} | features: {len(features_existing)}')
if len(features_existing) == len(gt_wavs_existing) > 0:
    print('San sang train.')
else:
    print('CANH BAO: so files khong khop - kiem tra log trong logs/' + EXPERIMENT_NAME)


Device: cuda:0 | FP16: True
Preprocess: da co (212 clips) - bo qua
Trich F0 + HuBERT features...
infer/modules/train/extract/extract_f0_print.py /content/compare_rvc/logs/rvc_experiment 2 rmvpe
todo-f0-106
f0ing,now-0,all-106,-/content/compare_rvc/logs/rvc_experiment/1_16k_wavs/0_0.wav
f0ing,now-21,all-106,-/content/compare_rvc/logs/rvc_experiment/1_16k_wavs/26_4.wav
f0ing,now-42,all-106,-/content/compare_rvc/logs/rvc_experiment/1_16k_wavs/43_4.wav
f0ing,now-63,all-106,-/content/compare_rvc/logs/rvc_experiment/1_16k_wavs/60_1.wav
f0ing,now-84,all-106,-/content/compare_rvc/logs/rvc_experiment/1_16k_wavs/79_3.wav
f0ing,now-105,all-106,-/content/compare_rvc/logs/rvc_experiment/1_16k_wavs/93_3.wav
todo-f0-106
f0ing,now-0,all-106,-/content/compare_rvc/logs/rvc_experiment/1_16k_wavs/0_2.wav
f0ing,now-21,all-106,-/content/compare_rvc/logs/rvc_experiment/1_16k_wavs/27_1.wav
f0ing,now-42,all-106,-/content/compare_rvc/logs/rvc_experiment/1_16k_wavs/44_0.wav
f0ing,now-63,all-106,-/content/compare

RuntimeError: extract_feature_print.py thất bại (exit 1). Đọc output phía trên và file logs/rvc_experiment/extract_f0_feature.log — thường gặp: Hubert hỏng/thiếu, fairseq/torch lỗi, hoặc VRAM không đủ khi đưa model lên GPU.

## Cell 10 — Xây Dựng FAISS Index

**Mục đích:** Tạo FAISS index từ toàn bộ HuBERT feature vectors đã trích ở Cell 9.
Index này được dùng trong bước inference để tìm kiếm feature gần nhất từ tập training
và áp lên giọng target, giúp chuyển giọng tự nhiên hơn.

**Cơ chế:**
- Toàn bộ file `.npy` trong `3_feature768/` được gom thành một ma trận lớn.
- Thuật toán `IVF` (Inverted File Index) + `PQ` (Product Quantization) được dùng
  để nén và đánh index hiệu quả.
- Output: `logs/rvc_experiment/added_*.index` — file index cuối cùng dùng khi inference.

**Khi nào thì dùng index?**
Tham số `INFER_INDEX_RATE` ở Cell 4 điều khiển mức độ ảnh hưởng của index.
- `0.0` = bỏ qua hoàn toàn, chỉ dùng model
- `0.75` = kết hợp 75% từ index + 25% từ model
- `1.0` = chỉ dùng index (thường bị artifact)

Cell này chỉ chạy **một lần** — index không đổi dù train thêm epoch.
Nếu đã có file `added_*.index` trong `logs/rvc_experiment/` → bỏ qua.

Đặt `SKIP_INDEX = True` trong Cell 4 nếu muốn bỏ qua bước này hoàn toàn.


In [ ]:
exp_dir          = STANDALONE_ROOT / 'logs' / EXPERIMENT_NAME
existing_indexes = list(exp_dir.glob('added_*.index'))

if existing_indexes:
    print('FAISS index da co:')
    for f in existing_indexes:
        print(f'  {f.name}  ({f.stat().st_size/1024**2:.1f} MB)')
elif SKIP_INDEX:
    print('SKIP_INDEX=True - bo qua FAISS index')
else:
    print('Dang tao FAISS Index...')
    for line in train_steps.step_train_index(STANDALONE_ROOT, config, p):
        print(line)
    existing_indexes = list(exp_dir.glob('added_*.index'))
    if existing_indexes:
        print(f'Index tao xong: {existing_indexes[-1].name}')
    else:
        print('CANH BAO: khong tim thay index file sau khi tao')


## Cell 11 — Load Model Đánh Giá & Tính Baseline Embedding

**Mục đích:** Khởi tạo ba model đánh giá và tính trước các embedding cần thiết.
Làm một lần trước vòng lặp để không phải reload lại sau mỗi epoch.

### 1 — Speaker Embedding (SpeechBrain ECAPA-TDNN)
Model `spkrec-ecapa-voxceleb` encode audio thành vector 192 chiều đặc trưng cho giọng nói.
Cell tính trước:
- **`target_centroid`:** Vector trung bình của tất cả file `target_eval` — đây là
  "chân dung giọng" của target speaker, dùng làm chuẩn để đo similarity.
- **`source_embeddings`:** Dictionary `{filename: embedding}` cho từng file source test.

### 2 — ASR (Whisper)
Load pipeline `openai/whisper-small` để transcribe audio converted → tính WER/CER.
Cần thiết để đánh giá xem giọng sau khi chuyển có còn giữ được nội dung không.

### 3 — UTMOS (Predicted MOS)
Load model từ `tarepan/SpeechMOS` qua `torch.hub` để dự đoán điểm chất lượng âm thanh
theo thang 1–5 (Mean Opinion Score).
Nếu load thất bại (mạng chậm, version không tương thích) → MOS sẽ ghi `NaN`, pipeline vẫn tiếp tục.

### 4 — VC Pipeline
Khởi tạo `VC` object từ `infer/modules/vc/` và load HuBERT model vào GPU.
Object này được dùng lại cho tất cả inference trong vòng lặp chính.


In [ ]:
from speechbrain.inference.speaker import EncoderClassifier

SPEAKER_MODEL_DIR = str(PROJECT_DIR / 'pretrained_models' / 'spkrec-ecapa-voxceleb')
print('Load speaker encoder...')
speaker_encoder = EncoderClassifier.from_hparams(
    source='speechbrain/spkrec-ecapa-voxceleb',
    savedir=SPEAKER_MODEL_DIR,
    run_opts={'device': DEVICE},
)


def load_audio_16k(path: Path) -> torch.Tensor:
    wav, sr = torchaudio.load(str(path))
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)
    wav = wav.squeeze(0)
    return wav / wav.abs().max() if wav.abs().max() > 0 else wav


def speaker_embedding(path: Path) -> np.ndarray:
    wav = load_audio_16k(path).to(DEVICE)
    with torch.no_grad():
        emb = speaker_encoder.encode_batch(wav.unsqueeze(0))
    return emb.squeeze().detach().cpu().numpy().astype(np.float32)


def cos_sim(a: np.ndarray, b: np.ndarray) -> float:
    return float(cosine_similarity(a.reshape(1, -1), b.reshape(1, -1))[0, 0])


target_eval_manifest = pd.read_csv(PREPARED_DIR / 'target_eval_manifest.csv')
source_manifest_df   = pd.read_csv(PREPARED_DIR / 'source_test_manifest.csv')

print('Tinh target embedding centroid...')
target_centroid = np.mean(
    np.vstack([speaker_embedding(Path(r))
               for r in tqdm(target_eval_manifest['path'], desc='Target emb')]),
    axis=0
)

print('Tinh source embeddings...')
source_embeddings: dict = {}
for _, row in tqdm(source_manifest_df.iterrows(), total=len(source_manifest_df)):
    source_embeddings[row['file']] = speaker_embedding(Path(row['path']))
print(f'Target centroid + {len(source_embeddings)} source embeddings san sang')

from transformers import pipeline as hf_pipeline

print(f'Load ASR {ASR_MODEL_ID}...')
asr_pipe = hf_pipeline(
    'automatic-speech-recognition', model=ASR_MODEL_ID,
    device=0 if DEVICE == 'cuda' else -1,
)


def normalize_text(s: str) -> str:
    if s is None:
        return ''
    s = str(s).lower().strip()
    s = unicodedata.normalize('NFC', s)
    s = re.sub(r'[^\w\s]', ' ', s, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', s).strip()


def transcribe(path: Path) -> str:
    kw = {'generate_kwargs': {'language': ASR_LANGUAGE, 'task': 'transcribe'}} if ASR_LANGUAGE else {}
    r  = asr_pipe(str(path), **kw)
    return r.get('text', '') if isinstance(r, dict) else str(r)


mos_predictor = None
mos_backend_available = False

if RUN_PREDICTED_MOS and MOS_BACKEND == 'utmos_torchhub':
    print('Load UTMOS...')
    try:
        mos_predictor = torch.hub.load('tarepan/SpeechMOS:v1.2.0', 'utmos22_strong', trust_repo=True)
        if hasattr(mos_predictor, 'to'):
            mos_predictor = mos_predictor.to(DEVICE)
        if hasattr(mos_predictor, 'eval'):
            mos_predictor.eval()
        mos_backend_available = True
        print('UTMOS OK')
    except Exception as e:
        print(f'UTMOS load that bai: {e} - MOS se ghi NaN')


def predict_mos(path: Path) -> float:
    wav, sr = torchaudio.load(str(path))
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    wav = wav.to(DEVICE)
    with torch.no_grad():
        for fn in [lambda: mos_predictor(wav, sr), lambda: mos_predictor(wav.squeeze(0), sr)]:
            try:
                score = fn()
                if isinstance(score, (tuple, list)):
                    score = score[0]
                if isinstance(score, torch.Tensor):
                    score = score.detach().cpu().numpy()
                return float(np.asarray(score).reshape(-1)[0])
            except Exception:
                continue
    raise RuntimeError(f'Khong predict duoc MOS: {path}')


print('Load VC inference pipeline...')
_saved_argv = sys.argv
sys.argv    = ['rvc_infer']
try:
    from configs.config import Config as RVCConfig
    rvc_config = RVCConfig()
finally:
    sys.argv = _saved_argv

from infer.modules.vc.modules import VC
from infer.modules.vc.utils import load_hubert

vc_pipeline = VC(rvc_config)
vc_pipeline.hubert_model = load_hubert(rvc_config)
print('VC pipeline san sang')


## Cell 12 — Vòng Lặp Chính: Training → Export → Inference → Đánh Giá

**Mục đích:** Chạy toàn bộ pipeline lặp qua từng mốc epoch trong `EPOCH_LIST_EFFECTIVE`.
Mỗi vòng lặp thực hiện 4 bước:

### Bước 1 — Training
Gọi `step_train()` với `total_epochs = epoch_target`.
RVC tự động phát hiện checkpoint cuối trong `logs/rvc_experiment/` và **tiếp tục training**
thay vì train lại từ đầu — đây là lý do EPOCH_LIST phải tăng dần.

### Bước 2 — Export model nhỏ
Từ checkpoint G (Generator) mới nhất, trích ra model nhỏ gọn
lưu vào `assets/weights/model_<N>ep.pth` (~70 MB thay vì ~600 MB checkpoint đầy đủ).
File `.pth` này là thứ được dùng khi inference.

### Bước 3 — Inference
Với từng file trong `source_test_manifest.csv`, gọi `vc_single()` để chuyển giọng:
source audio → converted audio theo giọng target.
Kết quả lưu vào `Drive/eval_project/converted_output/model_<N>ep/`.

### Bước 4 — Đánh giá
Ba metric được tính cho mỗi file converted:
- **Speaker Similarity:** cosine similarity giữa embedding của converted và `target_centroid`.
  So sánh với similarity của source gốc để đo `improvement`.
- **Predicted MOS:** UTMOS dự đoán chất lượng âm thanh (1–5).
- **WER/CER:** Whisper transcribe converted audio, so sánh với transcript gốc.

Sau khi hoàn thành tất cả vòng lặp, ba DataFrame kết quả được lưu vào `Drive/results/`.

> **Ước tính thời gian tổng** với T4, EPOCH_LIST=[50,100,200]:
> training ~3–5h, inference+eval ~30–60 phút. Tổng ~4–6 giờ.
> Nếu session Colab reset giữa chừng, training sẽ resume từ checkpoint cuối.


In [ ]:
all_sim_rows: List[dict] = []
all_mos_rows: List[dict] = []
all_asr_rows: List[dict] = []

source_manifest_df = pd.read_csv(PREPARED_DIR / 'source_test_manifest.csv')
print(f'Se chay {len(EPOCH_LIST_EFFECTIVE)} model: {[f"model_{e}ep" for e in EPOCH_LIST_EFFECTIVE]}')
t_start = time.time()

for epoch_target in EPOCH_LIST_EFFECTIVE:
    model_name = f'model_{epoch_target}ep'
    t0         = time.time()
    print(f'\n{"="*60}\n  {model_name} | epoch {epoch_target}\n{"="*60}')

    print(f'[1/4] Training -> epoch {epoch_target}...')
    p.total_epochs      = epoch_target
    p.infer_weight_name = model_name
    train_steps.step_train(STANDALONE_ROOT, config, p)

    print(f'[2/4] Export -> assets/weights/{model_name}.pth')
    p.g_checkpoint_for_extract = ''
    print(train_steps.step_extract_small_weights(STANDALONE_ROOT, p))

    model_pth = STANDALONE_ROOT / 'assets' / 'weights' / f'{model_name}.pth'
    if not model_pth.exists():
        print(f'CANH BAO: {model_pth.name} khong tim thay - bo qua')
        continue
    print(f'  {model_pth.name} ({model_pth.stat().st_size/1024**2:.1f} MB)')

    print(f'[3/4] Inference -> converted_output/{model_name}/')
    vc_pipeline.get_vc(f'{model_name}.pth')
    exp_dir     = STANDALONE_ROOT / 'logs' / EXPERIMENT_NAME
    index_files = sorted(exp_dir.glob('added_*.index'), key=lambda f: f.stat().st_mtime)
    file_index  = str(index_files[-1]) if (index_files and not SKIP_INDEX and INFER_INDEX_RATE > 0) else ''
    out_dir     = CONVERTED_ROOT / model_name
    out_dir.mkdir(parents=True, exist_ok=True)

    for _, row in tqdm(source_manifest_df.iterrows(), total=len(source_manifest_df),
                       desc=f'Infer {model_name}'):
        try:
            _, audio_out = vc_pipeline.vc_single(
                INFER_SPEAKER_ID, row['path'], INFER_F0_UP_KEY, None,
                INFER_F0_METHOD, file_index, '', INFER_INDEX_RATE,
                INFER_FILTER_RADIUS, INFER_RESAMPLE_SR, INFER_RMS_MIX_RATE, INFER_PROTECT,
            )
            if audio_out:
                tgt_sr, audio_i16 = audio_out
                sf.write(str(out_dir / row['file']), audio_i16, tgt_sr)
        except Exception as e:
            print(f'  Loi infer {row["file"]}: {e}')
    print(f'  {len(list(out_dir.glob("*.wav")))}/{len(source_manifest_df)} files converted')

    print(f'[4/4] Danh gia {model_name}...')
    for _, row in tqdm(source_manifest_df.iterrows(), total=len(source_manifest_df), desc='Speaker Sim'):
        converted = out_dir / row['file']
        src_tgt   = cos_sim(source_embeddings[row['file']], target_centroid)
        base      = {'model': model_name, 'epoch': epoch_target, 'file': row['file']}
        if not converted.exists():
            all_sim_rows.append({**base, 'source_target_similarity': src_tgt,
                'converted_target_similarity': np.nan, 'improvement': np.nan, 'converted_exists': False})
        else:
            conv_tgt = cos_sim(speaker_embedding(converted), target_centroid)
            all_sim_rows.append({**base, 'source_target_similarity': src_tgt,
                'converted_target_similarity': conv_tgt,
                'improvement': conv_tgt - src_tgt, 'converted_exists': True})

    for _, row in tqdm(source_manifest_df.iterrows(), total=len(source_manifest_df), desc='MOS'):
        converted = out_dir / row['file']
        base      = {'model': model_name, 'epoch': epoch_target, 'file': row['file']}
        if not converted.exists():
            all_mos_rows.append({**base, 'predicted_mos': np.nan, 'mos_error': 'missing'})
        elif not (RUN_PREDICTED_MOS and mos_backend_available):
            all_mos_rows.append({**base, 'predicted_mos': np.nan, 'mos_error': 'unavailable'})
        else:
            try:
                all_mos_rows.append({**base, 'predicted_mos': predict_mos(converted), 'mos_error': ''})
            except Exception as e:
                all_mos_rows.append({**base, 'predicted_mos': np.nan, 'mos_error': str(e)})

    for _, row in tqdm(source_manifest_df.iterrows(), total=len(source_manifest_df), desc='ASR WER'):
        converted = out_dir / row['file']
        ref  = normalize_text(row['transcript'])
        base = {'model': model_name, 'epoch': epoch_target, 'file': row['file'], 'reference': ref}
        if not converted.exists():
            all_asr_rows.append({**base, 'hypothesis': '', 'wer': np.nan, 'cer': np.nan, 'asr_error': 'missing'})
        else:
            try:
                hyp = normalize_text(transcribe(converted))
                all_asr_rows.append({**base, 'hypothesis': hyp,
                    'wer': jiwer.wer(ref, hyp) if ref else np.nan,
                    'cer': jiwer.cer(ref, hyp) if ref else np.nan, 'asr_error': ''})
            except Exception as e:
                all_asr_rows.append({**base, 'hypothesis': '', 'wer': np.nan,
                    'cer': np.nan, 'asr_error': str(e)})

    print(f'{model_name} xong: {(time.time()-t0)/60:.0f} phut')

sim_df = pd.DataFrame(all_sim_rows)
mos_df = pd.DataFrame(all_mos_rows)
asr_df = pd.DataFrame(all_asr_rows)

for df, name in [(sim_df, 'speaker_similarity'), (mos_df, 'predicted_mos'), (asr_df, 'asr_wer_cer')]:
    if not df.empty:
        path = RESULTS_DIR / f'{name}_results.csv'
        df.to_csv(path, index=False, encoding='utf-8-sig')
        print(f'Luu: {path}')

print(f'HOAN THANH: {(time.time()-t_start)/3600:.1f} gio')


## Cell 13 — Tổng Hợp Kết Quả & Vẽ Biểu Đồ So Sánh

**Mục đích:** Đọc kết quả từ ba file CSV (similarity, MOS, ASR) đã lưu ở Cell 12,
merge lại thành một bảng tổng hợp và vẽ biểu đồ so sánh các model.

**Cell này có thể chạy độc lập** sau khi vòng lặp chính đã hoàn thành,
hoặc chạy lại bất cứ lúc nào để xem lại kết quả từ Drive.

**Output:**

| File | Nội dung |
|------|---------|
| `per_sample_results.csv` | Kết quả chi tiết từng file audio cho từng model |
| `summary_by_model.csv` | Giá trị trung bình mỗi metric theo từng model/epoch |
| `summary_plot.png` | Biểu đồ bar chart so sánh 4 metric |

**Cách đọc bảng tổng hợp:**

| Cột | Ý nghĩa | Hướng tốt |
|-----|---------|----------|
| `converted_target_similarity` | Cosine similarity giữa giọng converted và target | ↑ cao |
| `improvement` | Tăng so với similarity của source gốc | ↑ cao |
| `predicted_mos` | Chất lượng âm thanh dự đoán (1–5) | ↑ cao |
| `wer` | Word Error Rate | ↓ thấp |
| `cer` | Character Error Rate | ↓ thấp |

Tất cả file đều nằm tại `MyDrive/rvc_project/eval_project/results/`.


In [ ]:
def load_csv_if_exists(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


sim_df = load_csv_if_exists(RESULTS_DIR / 'speaker_similarity_results.csv')
mos_df = load_csv_if_exists(RESULTS_DIR / 'predicted_mos_results.csv')
asr_df = load_csv_if_exists(RESULTS_DIR / 'asr_wer_cer_results.csv')

per_sample = sim_df.copy() if not sim_df.empty else None
if per_sample is not None and not mos_df.empty:
    per_sample = per_sample.merge(
        mos_df[['model', 'file', 'predicted_mos', 'mos_error']], on=['model', 'file'], how='left')
if per_sample is not None and not asr_df.empty:
    per_sample = per_sample.merge(
        asr_df[['model', 'file', 'reference', 'hypothesis', 'wer', 'cer', 'asr_error']],
        on=['model', 'file'], how='left')

if per_sample is None or per_sample.empty:
    print('Chua co ket qua. Chay Cell 16 truoc.')
else:
    per_sample.to_csv(RESULTS_DIR / 'per_sample_results.csv', index=False, encoding='utf-8-sig')

    agg_dict   = {'source_target_similarity': 'mean', 'converted_target_similarity': 'mean',
                  'improvement': 'mean'}
    group_cols = ['model', 'epoch'] if 'epoch' in per_sample.columns else ['model']
    for col in ['predicted_mos', 'wer', 'cer']:
        if col in per_sample.columns:
            agg_dict[col] = 'mean'

    summary             = per_sample.groupby(group_cols).agg(agg_dict).reset_index()
    summary['n_samples'] = per_sample.groupby(group_cols)['file'].count().values
    if 'epoch' in summary.columns:
        summary = summary.sort_values('epoch')

    summary.to_csv(RESULTS_DIR / 'summary_by_model.csv', index=False, encoding='utf-8-sig')

    print('BANG TONG HOP:')
    try:
        display(summary)
    except NameError:
        print(summary.to_string())

    metrics = [
        ('converted_target_similarity', 'Speaker Similarity (up)'),
        ('predicted_mos',               'Predicted MOS (up)'),
        ('wer',                         'WER (down)'),
        ('cer',                         'CER (down)'),
    ]
    active = [(m, t) for m, t in metrics
              if m in summary.columns and not summary[m].isna().all()]

    if active:
        fig, axes = plt.subplots(1, len(active), figsize=(4 * len(active), 4), squeeze=False)
        x_labels  = summary['model'].tolist()
        for ax_i, (metric, title) in enumerate(active):
            ax = axes[0][ax_i]
            ax.bar(x_labels, summary[metric], color='steelblue', edgecolor='white')
            ax.set_title(title, fontsize=11)
            ax.set_xlabel('Model')
            ax.tick_params(axis='x', rotation=30)
            for bar, val in zip(ax.patches, summary[metric]):
                if not np.isnan(val):
                    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                            f'{val:.3f}', ha='center', va='bottom', fontsize=9)
        plt.tight_layout()
        out_plot = RESULTS_DIR / 'summary_plot.png'
        plt.savefig(out_plot, dpi=160, bbox_inches='tight')
        plt.show()

    for f in ['per_sample_results.csv', 'summary_by_model.csv', 'summary_plot.png']:
        p_csv = RESULTS_DIR / f
        if p_csv.exists():
            print(f'{p_csv}')


## Hướng Dẫn Resume & Điều Chỉnh

### Resume sau khi Colab reset

Vì checkpoint và data đã trên Drive, việc resume chỉ cần:

```
Cell 1  — Mount Drive + GPU check
Cell 2  — Clone repo (git pull nếu đã có)
Cell 3  — Tạo lại symlink về Drive
Cell 2b — Cài thư viện (bắt buộc mỗi session)
Cell 4  — Load config
Cell 5  — Env check (tuỳ chọn)
Cell 7 — Import + setup
Cell 9 — Training setup (tự phát hiện preprocess đã có, bỏ qua)
Cell 10 — FAISS index (bỏ qua nếu đã có)
Cell 11 — Load eval models
Cell 12 — Tiếp tục vòng lặp (training tự resume từ checkpoint cuối)
```

### Điều chỉnh nếu kết quả kém

| Triệu chứng | Nguyên nhân có thể | Cách khắc phục |
|------------|-------------------|----------------|
| Similarity thấp (<0.7) | Chưa đủ epoch hoặc data | Tăng `EPOCH_LIST` hoặc `TARGET_TRAIN_SECONDS` |
| WER/CER cao | Index quá mạnh, mất nội dung | Giảm `INFER_INDEX_RATE` xuống 0.3–0.5 |
| MOS thấp (<3.5) | Artifact từ inference | Tăng `INFER_PROTECT`, giảm `INFER_RMS_MIX_RATE` |
| Giọng sai tông | Khác giới tính source/target | Đặt `INFER_F0_UP_KEY = +12` hoặc `-12` |
| OOM trên T4 | Batch size quá lớn | Giảm `BATCH_SIZE` xuống 4 |

### Cấu trúc thư mục trên Drive sau khi chạy xong

```
MyDrive/rvc_project/
├── logs/rvc_experiment/          # Checkpoint, features, F0
├── model_weights/                # Exported .pth files
├── assets_cache/                 # HuBERT, RMVPE, pretrained
└── eval_project/
    ├── raw_data/LibriTTS/        # Dataset gốc
    ├── prepared_data/            # WAV chuẩn hóa + manifest CSV
    ├── converted_output/         # Audio sau khi chuyển giọng
    └── results/                  # CSV kết quả + biểu đồ
```
